In [1]:
import re
from datetime import datetime, timedelta

class DualLearningAnalyzer:
    def __init__(self, session_gap_minutes=20, active_threshold_logs=3):
        """
        :param session_gap_minutes: 超过多少分钟没有操作，就切分为下一个独立会话
        :param active_threshold_logs: 一个会话内最少需要的日志条数，少于该值视为疑似挂机
        """
        self.session_gap = timedelta(minutes=session_gap_minutes)
        self.active_threshold = active_threshold_logs

    def parse_logs(self, log_text):
        """核心：解析日志并提取时间戳"""
        pattern = r'\[[A-Z]\s(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{3})\sServerApp\] (.*)'
        matches = re.findall(pattern, log_text)
        
        logs = []
        for ts_str, msg in matches:
            dt = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")
            logs.append({"time": dt, "msg": msg.lower()})
        return sorted(logs, key=lambda x: x["time"])

    def calculate_baseline(self, logs, threshold_minutes=5):
        """算法一：基础邻近间隔法（需求3）"""
        if len(logs) < 2:
            return timedelta()
        
        total_duration = timedelta()
        threshold = timedelta(minutes=threshold_minutes)
        
        for i in range(len(logs) - 1):
            diff = logs[i+1]["time"] - logs[i]["time"]
            if diff <= threshold:
                total_duration += diff
        return total_duration

    def calculate_advanced_and_sessions(self, logs):
        """算法二：高级演算法（会话切片 + 挂机过滤 + 尾部补偿）"""
        if not logs:
            return timedelta(), []

        # 1. 会话切片
        sessions = []
        current_session = [logs[0]]

        for log in logs[1:]:
            if log["time"] - current_session[-1]["time"] > self.session_gap:
                sessions.append(current_session)
                current_session = [log]
            else:
                current_session.append(log)
        sessions.append(current_session)

        # 2. 精细化计算
        total_adv_time = timedelta()
        session_reports = []

        for idx, session in enumerate(sessions, 1):
            start_time = session[0]["time"]
            end_time = session[-1]["time"]
            raw_duration = end_time - start_time
            log_count = len(session)
            
            has_kernel_activity = any("kernel" in l["msg"] for l in session)
            
            # 智能挂机识别
            if log_count < self.active_threshold and not has_kernel_activity:
                adjusted_duration = timedelta(minutes=2) * log_count
                status = "疑似挂机/短暂离开"
            else:
                last_msg = session[-1]["msg"]
                # 尾部补偿：最后一次操作后给予 5-10 分钟思考留存
                tail_buffer = timedelta(minutes=10) if "saving file" in last_msg else timedelta(minutes=5)
                adjusted_duration = raw_duration + tail_buffer
                status = "高效专注"

            total_adv_time += adjusted_duration
            
            session_reports.append({
                "session_id": idx,
                "start": start_time.strftime("%H:%M:%S"),
                "end": end_time.strftime("%H:%M:%S"),
                "log_count": log_count,
                "status": status,
                "duration": adjusted_duration
            })

        return total_adv_time, session_reports

    def run_analysis(self, log_text):
        logs = self.parse_logs(log_text)
        if not logs:
            print("❌ 未检测到有效日志数据。")
            return

        date_str = logs[0]["time"].strftime("%Y-%m-%d")
        
        # 计算两种算法
        base_time = self.calculate_baseline(logs, threshold_minutes=5)
        adv_time, sessions = self.calculate_advanced_and_sessions(logs)
        
        # 转换为分钟方便对比
        base_min = base_time.total_seconds() / 60
        adv_min = adv_time.total_seconds() / 60
        diff_percent = ((adv_min - base_min) / base_min * 100) if base_min > 0 else 0

        # ==================== 打印对比看板 ====================
        print(f"=== 🧠 双算法对比学习看板 ({date_str}) ===")
        print(f"{'计算维度':<20}{'基础算法 (5分钟间隔)':<25}{'高级智能算法 (会话演进)'}")
        print("-" * 75)
        print(f"{'总统计时长':<17}{base_min:>10.2f} 分钟{adv_min:>22.2f} 分钟")
        print(f"{'算法核心逻辑':<15}{'仅加 <5min 的间隔':<22}{'动态Buffer + 挂机剔除'}")
        print(f"{'算法差异率':<17}{'---':>12}{diff_percent:>20.1f}% (高级算法比基础算法)")
        print("-" * 75)
        print("\n🔍 高级算法拆解的课时明细：")
        print("-" * 75)
        print(f"{'课时':<6}{'开始时间':<10}{'结束时间':<10}{'操作数':<8}{'状态':<18}{'计算时长'}")
        print("-" * 75)
        for s in sessions:
            s_min = s['duration'].total_seconds() / 60
            print(f"#{s['session_id']:<5}{s['start']:<10}{s['end']:<10}{s['log_count']:<9}{s['status']:<18}{s_min:.1f} 分钟")
        print("-" * 75)


# ==================== 测试数据运行 ====================
if __name__ == "__main__":
    # 包含密集的学习操作（11:10~11:17），以及28分钟后的一条孤立自动保存日志（11:45 模拟挂机离开）
    raw_logs = """
    [I 2026-07-03 09:34:39.241 ServerApp] jupyter_lsp | extension was successfully linked.
[I 2026-07-03 09:34:39.254 ServerApp] jupyter_server_terminals | extension was successfully linked.
[I 2026-07-03 09:34:39.270 ServerApp] jupyterlab | extension was successfully linked.
[I 2026-07-03 09:34:39.282 ServerApp] notebook | extension was successfully linked.
[I 2026-07-03 09:34:40.026 ServerApp] notebook_shim | extension was successfully linked.
[I 2026-07-03 09:34:40.083 ServerApp] notebook_shim | extension was successfully loaded.
[I 2026-07-03 09:34:40.087 ServerApp] jupyter_lsp | extension was successfully loaded.
[I 2026-07-03 09:34:40.091 ServerApp] jupyter_server_terminals | extension was successfully loaded.
[I 2026-07-03 09:34:40.100 LabApp] JupyterLab extension loaded from C:\nitamade\envs\d2l\lib\site-packages\jupyterlab
[I 2026-07-03 09:34:40.100 LabApp] JupyterLab application directory is C:\nitamade\envs\d2l\share\jupyter\lab
[I 2026-07-03 09:34:40.100 LabApp] Extension Manager is 'pypi'.
[I 2026-07-03 09:34:40.220 ServerApp] jupyterlab | extension was successfully loaded.
[I 2026-07-03 09:34:40.233 ServerApp] notebook | extension was successfully loaded.
[I 2026-07-03 09:34:40.237 ServerApp] Serving notebooks from local directory: d:\github
[I 2026-07-03 09:34:40.237 ServerApp] Jupyter Server 2.19.0 is running at:
[I 2026-07-03 09:34:40.237 ServerApp] http://localhost:8888/tree?token=0112bb9a0b9e465ddba230e4197b683a3855103e90ba5bc8
[I 2026-07-03 09:34:40.237 ServerApp]     http://127.0.0.1:8888/tree?token=0112bb9a0b9e465ddba230e4197b683a3855103e90ba5bc8
[I 2026-07-03 09:34:40.237 ServerApp] Use Control-C to stop this server and shut down all kernels (twice to skip confirmation).
[C 2026-07-03 09:34:40.382 ServerApp]

    To access the server, open this file in a browser:
        file:///C:/Users/Administrator/AppData/Roaming/jupyter/runtime/jpserver-9236-open.html
    Or copy and paste one of these URLs:
        http://localhost:8888/tree?token=0112bb9a0b9e465ddba230e4197b683a3855103e90ba5bc8
        http://127.0.0.1:8888/tree?token=0112bb9a0b9e465ddba230e4197b683a3855103e90ba5bc8
[I 2026-07-03 09:34:40.632 ServerApp] Skipped non-installed server(s): basedpyright, bash-language-server, dockerfile-language-server-nodejs, javascript-typescript-langserver, jedi-language-server, julia-language-server, pyrefly, pyright, python-language-server, python-lsp-server, r-languageserver, sql-language-server, texlab, typescript-language-server, unified-language-server, vscode-css-languageserver-bin, vscode-html-languageserver-bin, vscode-json-languageserver-bin, yaml-language-server
[I 2026-07-03 09:34:53.982 ServerApp] Kernel started: 09bb7069-c25f-4fc4-9e0c-38e844aa1779
[E 2026-07-03 09:34:54.345 ServerApp] Uncaught exception GET /api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=7a050a06-7f32-4660-894b-cc9a4f667aba (::1)
    HTTPServerRequest(protocol='http', host='localhost:8888', method='GET', uri='/api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=7a050a06-7f32-4660-894b-cc9a4f667aba', version='HTTP/1.1', remote_ip='::1')
    Traceback (most recent call last):
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\web.py", line 1880, in _execute
        result = await result
      File "C:\nitamade\envs\d2l\lib\site-packages\jupyter_server\services\kernels\websocket.py", line 66, in get
        await super().get(kernel_id=kernel_id)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 277, in get
        await self.ws_connection.accept_connection(self)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 890, in accept_connection
        await self._accept_connection(handler)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 930, in _accept_connection
        self.selected_subprotocol = handler.select_subprotocol(subprotocols)
      File "C:\nitamade\envs\d2l\lib\site-packages\jupyter_server\services\kernels\websocket.py", line 88, in select_subprotocol
        preferred_protocol = self.connection.kernel_ws_protocol
    AttributeError: 'NoneType' object has no attribute 'kernel_ws_protocol'
[E 2026-07-03 09:34:54.377 ServerApp] {
      "Host": "localhost:8888",
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
    }
[E 2026-07-03 09:34:54.377 ServerApp] 500 GET /api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=7a050a06-7f32-4660-894b-cc9a4f667aba (3da880b2aa1243b79f2edea020e2bbcb@::1) 376.64ms referer=None
[E 2026-07-03 09:34:54.378 ServerApp] Uncaught exception GET /api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=e00b8755-ae55-4409-8995-89383891d50b (127.0.0.1)
    HTTPServerRequest(protocol='http', host='localhost:8888', method='GET', uri='/api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=e00b8755-ae55-4409-8995-89383891d50b', version='HTTP/1.1', remote_ip='127.0.0.1')
    Traceback (most recent call last):
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\web.py", line 1880, in _execute
        result = await result
      File "C:\nitamade\envs\d2l\lib\site-packages\jupyter_server\services\kernels\websocket.py", line 66, in get
        await super().get(kernel_id=kernel_id)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 277, in get
        await self.ws_connection.accept_connection(self)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 890, in accept_connection
        await self._accept_connection(handler)
      File "C:\nitamade\envs\d2l\lib\site-packages\tornado\websocket.py", line 930, in _accept_connection
        self.selected_subprotocol = handler.select_subprotocol(subprotocols)
      File "C:\nitamade\envs\d2l\lib\site-packages\jupyter_server\services\kernels\websocket.py", line 88, in select_subprotocol
        preferred_protocol = self.connection.kernel_ws_protocol
    AttributeError: 'NoneType' object has no attribute 'kernel_ws_protocol'
[E 2026-07-03 09:34:54.380 ServerApp] {
      "Host": "localhost:8888",
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
    }
[E 2026-07-03 09:34:54.380 ServerApp] 500 GET /api/kernels/09bb7069-c25f-4fc4-9e0c-38e844aa1779/channels?session_id=e00b8755-ae55-4409-8995-89383891d50b (3da880b2aa1243b79f2edea020e2bbcb@127.0.0.1) 63.50ms referer=None
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[I 2026-07-03 09:55:46.403 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:55:46.406 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[W 2026-07-03 09:55:46.418 ServerApp] The websocket_ping_timeout (90000) cannot be longer than the websocket_ping_interval (30000).
    Setting websocket_ping_timeout=30000
[I 2026-07-03 09:55:50.745 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:55:50.746 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 09:55:51.154 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:55:51.158 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 09:56:03.473 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:56:03.474 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 09:56:03.981 ServerApp] Kernel started: bc187a56-d5c6-434a-a149-fc132f2b781e
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[I 2026-07-03 09:56:05.136 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 09:56:05.137 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 09:58:27.006 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:58:27.007 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 09:58:27.032 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 09:58:27.033 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 09:59:04.361 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 09:59:04.361 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 09:59:04.391 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 09:59:04.393 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:00:04.198 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:00:08.887 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:00:08.889 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:00:08.914 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:00:08.915 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:00:09.507 ServerApp] Kernel started: de51f55a-fa6c-4939-818f-d0f290e32dd9
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[I 2026-07-03 10:00:11.128 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:00:11.131 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:02:04.255 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:04:04.314 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:05:57.503 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:c72f4f26-9914-4cc0-af91-866998e40e33
[I 2026-07-03 10:06:05.598 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:06:05.600 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:06:05.628 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:06:05.630 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:06:05.650 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:06:05.655 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:06:05.787 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:0694015d-8ee3-409d-a469-3528d997d97b
[I 2026-07-03 10:06:14.026 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:06:14.027 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:06:14.055 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:06:14.056 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:06:14.087 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:06:14.089 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:06:14.207 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:15080898-5e4e-45e4-baa2-48681ac575dc
[I 2026-07-03 10:07:46.302 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:07:46.303 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:07:46.332 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:07:46.334 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:07:46.357 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:07:46.359 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:07:46.582 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:2a85a1ac-48a8-4cf6-8bcc-f625ab3118ff
[I 2026-07-03 10:07:46.775 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:07:46.776 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:11:37.614 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:13:50.173 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:13:52.108 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:17:50.087 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:29:17.962 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:29:18.944 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:30:18.746 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:e62a80dd-7b08-46ea-b8bd-6692d4a943ec
[I 2026-07-03 10:30:26.121 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:30:26.122 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:30:26.166 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:30:26.166 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:30:26.193 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:30:26.197 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:30:26.390 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:f880a0a5-5072-4859-8e32-9814fdaf5f9e
[I 2026-07-03 10:30:27.482 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:30:27.482 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:30:27.513 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:30:27.514 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:30:27.543 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:30:27.547 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:30:27.716 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:d67133d9-d12c-449f-bbdd-2143c6143124
[I 2026-07-03 10:31:19.002 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:33:19.224 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:33:52.283 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:33:52.283 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:33:52.305 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:33:52.307 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:33:52.339 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:33:52.341 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:33:52.504 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:6716e8f7-9c41-4917-908e-86ff99503bef
[I 2026-07-03 10:35:20.203 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:36:49.753 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:36:49.755 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:36:49.784 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:36:49.787 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:36:49.813 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:36:49.815 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:36:50.103 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:fa229593-d7e5-49c5-b655-34b0c9c40967
[I 2026-07-03 10:36:50.230 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:36:50.232 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:37:21.237 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/pytorch_test_3.ipynb
[I 2026-07-03 10:39:46.317 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:39:46.319 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:39:46.348 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:39:46.348 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:39:46.375 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:39:46.382 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:39:49.864 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:67a8452f-491c-4674-b185-45e4f2926831
[I 2026-07-03 10:55:23.169 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/overfit_dropout.py
[I 2026-07-03 10:55:25.182 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/overfit_dropout.py
[I 2026-07-03 10:55:50.399 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:55:50.400 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:55:50.441 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:55:50.447 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:55:50.681 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:55:50.709 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:55:50.882 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:29141993-f61f-4d40-b6c2-d276ec379059
[I 2026-07-03 10:55:52.347 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:55:52.348 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:55:52.370 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:55:52.373 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:55:52.394 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:55:52.398 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:55:52.746 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:6ddfe184-5ba1-4747-a170-c39086872a74
[I 2026-07-03 10:56:27.267 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:56:27.268 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:56:27.288 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:56:27.291 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:56:27.320 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:56:27.322 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:56:27.546 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:dedd6571-cac0-4fdf-af6b-f610f8991b47
[I 2026-07-03 10:56:27.737 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:56:27.739 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:57:05.532 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/overfit_dropout.py
[I 2026-07-03 10:57:06.175 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch06/overfit_dropout.py
[I 2026-07-03 10:58:28.193 ServerApp] Saving file at /sakurantic-python-file/learningtime_calculator.ipynb
[I 2026-07-03 10:58:43.347 ServerApp] Starting buffering for bc187a56-d5c6-434a-a149-fc132f2b781e:9d735b84-99aa-491e-b09e-6b65647200ff
[I 2026-07-03 10:58:56.661 ServerApp] Creating new notebook in /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07
[I 2026-07-03 10:58:56.735 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/Untitled.ipynb
[I 2026-07-03 10:58:56.780 ServerApp] Kernel started: 69e062b5-b35c-44bc-ae41-a27d1c1c39ea
[I 2026-07-03 10:58:57.740 ServerApp] Adapting from protocol version 5.3 (kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779) to 5.4 (client).
[I 2026-07-03 10:58:57.743 ServerApp] Connecting to kernel 09bb7069-c25f-4fc4-9e0c-38e844aa1779.
[I 2026-07-03 10:58:57.832 ServerApp] Adapting from protocol version 5.3 (kernel bc187a56-d5c6-434a-a149-fc132f2b781e) to 5.4 (client).
[I 2026-07-03 10:58:57.838 ServerApp] Connecting to kernel bc187a56-d5c6-434a-a149-fc132f2b781e.
[I 2026-07-03 10:58:57.867 ServerApp] Adapting from protocol version 5.3 (kernel de51f55a-fa6c-4939-818f-d0f290e32dd9) to 5.4 (client).
[I 2026-07-03 10:58:57.876 ServerApp] Connecting to kernel de51f55a-fa6c-4939-818f-d0f290e32dd9.
[I 2026-07-03 10:58:57.936 ServerApp] Starting buffering for bc187a56-d5c6-434a-a149-fc132f2b781e:13bb2e11-9747-436a-a9d3-3bd77ce76549
[I 2026-07-03 10:58:57.946 ServerApp] Starting buffering for de51f55a-fa6c-4939-818f-d0f290e32dd9:5a61bfcb-2afb-4c88-8555-5baaeaddd190
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[I 2026-07-03 10:58:58.335 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 10:58:58.335 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 10:58:58.337 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 10:58:58.345 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 10:58:58.358 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 10:58:58.361 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[W 2026-07-03 10:58:58.399 ServerApp] Got events for closed stream <zmq.eventloop.zmqstream.ZMQStream object at 0x00000196BEC5AEF0>
[I 2026-07-03 11:01:32.207 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 11:03:33.200 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 11:11:36.219 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 11:13:37.199 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 11:23:39.828 ServerApp] Starting buffering for 69e062b5-b35c-44bc-ae41-a27d1c1c39ea:0b79a0a4-1d63-4273-883a-6a92bbb24e8b
[I 2026-07-03 11:25:34.178 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 11:25:34.181 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 11:25:34.198 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 11:25:34.198 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 11:47:57.990 ServerApp] Starting buffering for 69e062b5-b35c-44bc-ae41-a27d1c1c39ea:bdc2d6b2-9353-4854-a133-e83cc50d785a
[I 2026-07-03 11:48:54.177 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 11:48:54.178 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 11:48:55.181 ServerApp] Adapting from protocol version 5.3 (kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea) to 5.4 (client).
[I 2026-07-03 11:48:55.181 ServerApp] Connecting to kernel 69e062b5-b35c-44bc-ae41-a27d1c1c39ea.
[I 2026-07-03 11:57:18.616 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:02:25.063 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:10:24.691 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:16:37.397 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:16:38.325 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:20:29.134 ServerApp] Saving file at /pytorch/d2l-zh/deep-learning-from-scratch-master/ch07/cnn_test.ipynb
[I 2026-07-03 12:20:38.775 ServerApp] Interrupted...
[IPKernelApp] WARNING | Parent appears to have exited, shutting down.
[IPKernelApp] WARNING | Parent appears to have exited, shutting down.
[IPKernelApp] WARNING | Parent appears to have exited, shutting down.
[IPKernelApp] WARNING | Parent appears to have exited, shutting down.

(d2l) d:\github>jupyter notebook
[I 2026-07-03 12:21:17.980 ServerApp] jupyter_lsp | extension was successfully linked.
[I 2026-07-03 12:21:17.989 ServerApp] jupyter_server_terminals | extension was successfully linked.
[I 2026-07-03 12:21:18.003 ServerApp] jupyterlab | extension was successfully linked.
[I 2026-07-03 12:21:18.016 ServerApp] notebook | extension was successfully linked.
[I 2026-07-03 12:21:18.608 ServerApp] notebook_shim | extension was successfully linked.
[I 2026-07-03 12:21:18.653 ServerApp] notebook_shim | extension was successfully loaded.
[I 2026-07-03 12:21:18.657 ServerApp] jupyter_lsp | extension was successfully loaded.
[I 2026-07-03 12:21:18.657 ServerApp] jupyter_server_terminals | extension was successfully loaded.
[I 2026-07-03 12:21:18.669 LabApp] JupyterLab extension loaded from C:\nitamade\envs\d2l\lib\site-packages\jupyterlab
[I 2026-07-03 12:21:18.669 LabApp] JupyterLab application directory is C:\nitamade\envs\d2l\share\jupyter\lab
[I 2026-07-03 12:21:18.669 LabApp] Extension Manager is 'pypi'.
[I 2026-07-03 12:21:18.777 ServerApp] jupyterlab | extension was successfully loaded.
[I 2026-07-03 12:21:18.788 ServerApp] notebook | extension was successfully loaded.
[I 2026-07-03 12:21:18.789 ServerApp] Serving notebooks from local directory: d:\github
[I 2026-07-03 12:21:18.789 ServerApp] Jupyter Server 2.19.0 is running at:
[I 2026-07-03 12:21:18.789 ServerApp] http://localhost:8888/tree?token=3f989d6245a9c70e7829b18d0ce16d3079f6aca26fb7c194
[I 2026-07-03 12:21:18.789 ServerApp]     http://127.0.0.1:8888/tree?token=3f989d6245a9c70e7829b18d0ce16d3079f6aca26fb7c194
[I 2026-07-03 12:21:18.789 ServerApp] Use Control-C to stop this server and shut down all kernels (twice to skip confirmation).
[C 2026-07-03 12:21:18.910 ServerApp]

    To access the server, open this file in a browser:
        file:///C:/Users/Administrator/AppData/Roaming/jupyter/runtime/jpserver-16640-open.html
    Or copy and paste one of these URLs:
        http://localhost:8888/tree?token=3f989d6245a9c70e7829b18d0ce16d3079f6aca26fb7c194
        http://127.0.0.1:8888/tree?token=3f989d6245a9c70e7829b18d0ce16d3079f6aca26fb7c194
[I 2026-07-03 12:21:19.142 ServerApp] Skipped non-installed server(s): basedpyright, bash-language-server, dockerfile-language-server-nodejs, javascript-typescript-langserver, jedi-language-server, julia-language-server, pyrefly, pyright, python-language-server, python-lsp-server, r-languageserver, sql-language-server, texlab, typescript-language-server, unified-language-server, vscode-css-languageserver-bin, vscode-html-languageserver-bin, vscode-json-languageserver-bin, yaml-language-server
[I 2026-07-03 12:21:34.299 ServerApp] Kernel started: 217c3972-cbea-4eac-9e75-ae15136781de
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[I 2026-07-03 12:21:35.444 ServerApp] Adapting from protocol version 5.3 (kernel 217c3972-cbea-4eac-9e75-ae15136781de) to 5.4 (client).
[I 2026-07-03 12:21:35.444 ServerApp] Adapting from protocol version 5.3 (kernel 217c3972-cbea-4eac-9e75-ae15136781de) to 5.4 (client).
[I 2026-07-03 12:21:35.447 ServerApp] Connecting to kernel 217c3972-cbea-4eac-9e75-ae15136781de.
[I 2026-07-03 12:21:35.460 ServerApp] Connecting to kernel 217c3972-cbea-4eac-9e75-ae15136781de.
[W 2026-07-03 12:21:35.471 ServerApp] The websocket_ping_timeout (90000) cannot be longer than the websocket_ping_interval (30000).
    Setting websocket_ping_timeout=30000
[I 2026-07-03 12:21:35.475 ServerApp] Adapting from protocol version 5.3 (kernel 217c3972-cbea-4eac-9e75-ae15136781de) to 5.4 (client).
[I 2026-07-03 12:21:35.478 ServerApp] Connecting to kernel 217c3972-cbea-4eac-9e75-ae15136781de.
[W 2026-07-03 12:21:35.528 ServerApp] Got events for closed stream <zmq.eventloop.zmqstream.ZMQStream object at 0x000001E64B9C6B00>
[I 2026-07-03 12:21:41.179 ServerApp] Adapting from protocol version 5.3 (kernel 217c3972-cbea-4eac-9e75-ae15136781de) to 5.4 (client).
[I 2026-07-03 12:21:41.180 ServerApp] Connecting to kernel 217c3972-cbea-4eac-9e75-ae15136781de.
    """

    analyzer = DualLearningAnalyzer(session_gap_minutes=20, active_threshold_logs=3)
    analyzer.run_analysis(raw_logs)

=== 🧠 双算法对比学习看板 (2026-07-03) ===
计算维度                基础算法 (5分钟间隔)             高级智能算法 (会话演进)
---------------------------------------------------------------------------
总统计时长                 50.95 分钟                138.77 分钟
算法核心逻辑         仅加 <5min 的间隔          动态Buffer + 挂机剔除
算法差异率                     ---               172.4% (高级算法比基础算法)
---------------------------------------------------------------------------

🔍 高级算法拆解的课时明细：
---------------------------------------------------------------------------
课时    开始时间      结束时间      操作数     状态                计算时长
---------------------------------------------------------------------------
#1    09:34:39  09:34:54  23       高效专注              5.3 分钟
#2    09:55:46  11:25:34  158      高效专注              94.8 分钟
#3    11:47:57  12:21:41  39       高效专注              38.7 分钟
---------------------------------------------------------------------------
